Quiero que resuelvas este problema con python, con buena logica, logica simple, de la manera mas simple y facil de resolver.

Problema 1:

Dado un hash encontrar las secuencias de numeros que lo generan

reglas:
 1. Hash sha 256
 2. la secuencia de numeros son 10 enteros concatenados en el rango [0,9]

Problema 2:

Dado el hash root del arbol de merkel, determinar el orden de las transacciones que lo generan.

Reglas:
1. Las transacciones son conocidas, y el numero de transacciones


### Solución Problema 1: Fuerza Bruta SHA-256
Buscamos una secuencia de números (0-9) que genere un hash específico.

In [ ]:
import hashlib
from itertools import product

# Ingreso del hash
hash_objetivo = input("Ingresa el hash SHA-256 (10 dígitos): ").strip()

def resolver_problema_1(target):
    print(f"Buscando secuencia de 10 dígitos para: {target}...")

    # Probamos directamente todas las combinaciones de 10 dígitos
    for combo in product(range(10), repeat=10):
        secuencia = ''.join(map(str, combo))

        # Generamos hash y comparamos
        if hashlib.sha256(secuencia.encode()).hexdigest() == target:
            return secuencia
    return None

if hash_objetivo:
    resultado = resolver_problema_1(hash_objetivo)
    if resultado:
        print(f"¡Encontrado! La secuencia es: {resultado}")
    else:
        print("No se encontró la secuencia.")

Ingresa el hash SHA-256 (10 dígitos): 405a866e239e6ac94d2dab86612eaa1018b0d056951e5573da255ad28caaba64
Buscando secuencia de 10 dígitos para: 405a866e239e6ac94d2dab86612eaa1018b0d056951e5573da255ad28caaba64...
¡Encontrado! La secuencia es: 0000010000


### Solución Problema 2: Orden en Árbol de Merkle
Probamos el orden de las transacciones hasta que el hash de la raíz coincida.

In [ ]:
import hashlib
from itertools import permutations

def calcular_merkle_root(transacciones):
    # Hashear cada transacción individualmente (hojas)
    capa = [hashlib.sha256(str(tx).strip().encode()).hexdigest() for tx in transacciones]

    while len(capa) > 1:
        nueva_capa = []
        for i in range(0, len(capa), 2):
            hash1 = capa[i]
            if i + 1 < len(capa):
                # Si hay un par, se concatenan y hashean
                hash2 = capa[i+1]
                concat = hash1 + hash2
            else:
                # Si el nodo está solo (impar), se concatena consigo mismo
                concat = hash1 + hash1

            nueva_capa.append(hashlib.sha256(concat.encode()).hexdigest())
        capa = nueva_capa

    return capa[0] if capa else None

def encontrar_orden_merkle(transacciones_conocidas, root_objetivo):
    print(f"Buscando el orden correcto para el root: {root_objetivo}...")
    for p in permutations(transacciones_conocidas):
        if calcular_merkle_root(p) == root_objetivo:
            return p
    return None

# --- ENTRADA DE DATOS DINÁMICA ---
txs_input = input("Ingresa las transacciones conocidas separadas por coma (ej: A, B, C): ").strip()
root_input = input("Ingresa el Merkle Root objetivo: ").strip()

if txs_input and root_input:
    # Convertimos el string de entrada en una lista limpia
    txs_conocidas = [t.strip() for t in txs_input.split(",")]

    orden = encontrar_orden_merkle(txs_conocidas, root_input)
    if orden:
        print(f"¡Orden encontrado!: {orden}")
    else:
        print("No se encontró ningún orden que genere ese Merkle Root.")
else:
    print("Por favor, ingresa tanto las transacciones como el Root.")

Ingresa las transacciones conocidas separadas por coma (ej: A, B, C): E, F, G, C, D
Ingresa el Merkle Root objetivo: 83c3617c06f5580241dcbef3005b868c11f1faad94ad7cb2e096757fd049a66a
Buscando el orden correcto para el root: 83c3617c06f5580241dcbef3005b868c11f1faad94ad7cb2e096757fd049a66a...
No se encontró ningún orden que genere ese Merkle Root.


Funcion para calcular merkle root:

In [ ]:
import hashlib

def sha256(data):
    if isinstance(data, bytes):
        return hashlib.sha256(data).hexdigest()
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

def build_merkle_tree(data_nodes):
    if not data_nodes:
        return None

    # Calcular hashes de las hojas
    current_level = [sha256(node.strip()) for node in data_nodes]
    tree_levels = [current_level]

    while len(current_level) > 1:
        next_level = []
        for i in range(0, len(current_level), 2):
            hash1 = current_level[i]
            if i + 1 < len(current_level):
                # Si hay un par, se concatenan y hashean
                hash2 = current_level[i + 1]
                next_level.append(sha256(hash1 + hash2))
            else:
                next_level.append(hash1)

        current_level = next_level
        tree_levels.append(current_level)

    return {
        'leaves': tree_levels[0],
        'levels': tree_levels[1:],
        'root': current_level[0]
    }

# --- ENTRADA DEL USUARIO ---
txs_raw = input("Ingresa las transacciones (ej: A, B, C): ").strip()

if txs_raw:
    lista_txs = [t.strip() for t in txs_raw.split(",")]
    resultado = build_merkle_tree(lista_txs)

    print(f"\n--- Árbol de Merkle (Sin duplicar impares) ---")
    print(f"Hashes de las hojas: {resultado['leaves']}")
    for i, level in enumerate(resultado['levels']):
        print(f"Nivel {i+1}: {level}")
    print(f"Merkle Root Final: {resultado['root']}")
else:
    print("Por favor, ingresa al menos una transacción.")

In [ ]:

import hashlib

def sha256(data):
    if isinstance(data, bytes):
        return hashlib.sha256(data).hexdigest()
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

def build_merkle_tree(data_nodes):
    if not data_nodes:
        raise ValueError("No puede estar vacia.")

    # Compute leaf hashes
    current_level = [sha256(node) for node in data_nodes]
    tree_levels = [current_level]

    while len(current_level) > 1:
        next_level = []
        for i in range(0, len(current_level), 2):
            hash1 = current_level[i]
            hash2 = current_level[i + 1] if i + 1 < len(current_level) else hash1 # Handle odd number of hashes
            next_level.append(sha256(hash1 + hash2))
        current_level = next_level
        tree_levels.append(current_level)

    merkle_root = current_level[0]

    return {
        'leaves': tree_levels[0],
        'levels': tree_levels[1:],
        'root': merkle_root
    }

inputs_general_1 = ['hola', 'Hola', 'chao' 'Chao']
merkle_tree_general_1 = build_merkle_tree(inputs_general_1)

print("\n--- Merkle Tree for general inputs (6 items): ---")
print(f"  Original Inputs: {inputs_general_1}")
print(f"  Leaf Hashes: {merkle_tree_general_1['leaves']}")
for i, level in enumerate(merkle_tree_general_1['levels']):
    print(f"  Level {i+1} Hashes: {level}")
print(f"  Merkle Root: {merkle_tree_general_1['root']}")

inputs_general_2 = ['one', 'two', 'three'] # Example with odd number of inputs
merkle_tree_general_2 = build_merkle_tree(inputs_general_2)

print("\n--- Merkle Tree for general inputs (3 items): ---")
print(f"  Original Inputs: {inputs_general_2}")
print(f"  Leaf Hashes: {merkle_tree_general_2['leaves']}")
for i, level in enumerate(merkle_tree_general_2['levels']):
    print(f"  Level {i+1} Hashes: {level}")
print(f"  Merkle Root: {merkle_tree_general_2['root']}")